# Tutorial 11 — Constrained-Mixture Toy Model

This notebook connects constituent turnover, deposition stretch, fibrous-tissue loading phenotypes, and synthetic orientation-sensitive imaging-informed priors.


In [ ]:
LANGUAGE = "en"
from pathlib import Path
import sys


def _find_repository_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Repository root not found. Open the notebook inside the repository.")


REPOSITORY_ROOT = _find_repository_root(Path.cwd().resolve())
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import biomechanics_tutorials

print("Repository:", REPOSITORY_ROOT)
print("Package:", Path(biomechanics_tutorials.__file__).resolve())

import numpy as np
import matplotlib.pyplot as plt


from biomechanics_tutorials.constrained_mixture import (
    generic_loading_protocol,
    initialize_generic_constituents,
    mixture_mass_fractions,
    imaging_to_mixture_prior,
    simulate_constrained_mixture,
    simulate_homogenized_mixture,
)

## Initialize the model

All parameters are synthetic and normalized.


In [ ]:
time = np.linspace(0.0, 100.0, 251)
constituents = initialize_generic_constituents()
home = simulate_constrained_mixture(time, generic_loading_protocol("homeostasis"), constituents)
pressure = simulate_constrained_mixture(time, generic_loading_protocol("pressure"), constituents)
volume = simulate_constrained_mixture(time, generic_loading_protocol("volume"), constituents)

## Homeostasis and overload protocols


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(time, home["total_mass"], label="homeostasis")
axes[0].plot(time, pressure["total_mass"], label="pressure-like")
axes[0].plot(time, volume["total_mass"], label="volume-like")
axes[1].plot(time, home["total_stress"], label="homeostasis")
axes[1].plot(time, pressure["total_stress"], label="pressure-like")
axes[1].plot(time, volume["total_stress"], label="volume-like")
for ax in axes:
    ax.legend()
    ax.set_xlabel("time")
plt.show()

## Cardiomyocyte hypertrophy and collagen fibrosis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for result, label in [(pressure, "pressure-like"), (volume, "volume-like")]:
    myocyte = result["mass"]["cardiomyocytes"]
    collagen = result["mass"]["collagen_plus"] + result["mass"]["collagen_minus"]
    axes[0].plot(time, myocyte, label=label)
    axes[1].plot(time, collagen, label=label)
axes[0].set_title("cardiomyocytes")
axes[1].set_title("collagen")
for ax in axes:
    ax.legend()
    ax.set_xlabel("time")
plt.show()

## Full history versus homogenized reduction


In [ ]:
homogenized = simulate_homogenized_mixture(time, generic_loading_protocol("pressure"), constituents)
relative_error = np.abs(pressure["total_stress"] - homogenized["total_stress"]) / np.maximum(
    np.abs(pressure["total_stress"]), 1e-9
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(time, pressure["total_stress"], label="full history")
axes[0].plot(time, homogenized["total_stress"], "--", label="homogenized")
axes[1].plot(time, relative_error)
axes[0].legend()
axes[0].set_xlabel("time")
axes[1].set_xlabel("time")
axes[1].set_ylabel("relative error")
plt.show()

## Reversal and hereditary memory


In [ ]:
reversal = simulate_constrained_mixture(time, generic_loading_protocol("reversal"), constituents)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(time, pressure["total_mass"], label="sustained pressure-like")
axes[0].plot(time, reversal["total_mass"], label="load reversal")
axes[1].plot(time, pressure["total_stress"], label="sustained pressure-like")
axes[1].plot(time, reversal["total_stress"], label="load reversal")
for ax in axes:
    ax.legend()
    ax.set_xlabel("time")
plt.show()

## Synthetic orientation-sensitive imaging-to-prior bridge


In [ ]:
x = np.linspace(-1.0, 1.0, 100)
y = np.linspace(-1.0, 1.0, 80)
X, Y = np.meshgrid(x, y)
angle = 35.0 * np.sin(np.pi * X) * np.cos(0.5 * np.pi * Y)
beta = np.clip(0.25 + 0.7 * np.exp(-2.0 * (X**2 + Y**2)), 0.0, 1.0)
retardance = 0.3 + 0.7 * (X + 1.0) / 2.0
prior = imaging_to_mixture_prior(angle, beta, retardance)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(
    axes, [angle, beta, prior["collagen_fraction"]], ["angle", "order", "collagen prior"]
):
    image = ax.imshow(data, origin="lower", aspect="auto")
    fig.colorbar(image, ax=ax)
    ax.set_title(title)
plt.show()

## Final mass fractions


In [ ]:
fractions = mixture_mass_fractions(pressure)
{name: float(values[-1]) for name, values in fractions.items()}